# V8.1 Ablation Study: Biosemotic Features in Multilingual Laughter Detection
## XLM-R + Biosemotic Feature Ablation on en / zh / hi Stand-up Comedy

**Project:** `autonomous_laughter_prediction_essential`  
**Dataset:** `data/final_merged_10k/` — 10,048 multilingual examples  
**Backbone:** `FacebookAI/xlm-roberta-base`  
**Features:** 27 biosemotic dims (Duchenne, Incongruity, ToM, Cue, Structural, Linguistic, Metadata)

---

| # | Experiment | Description |
|---|------------|-------------|
| 1 | **Baseline XLM-R** | No biosemotic auxiliary features |
| 2 | **+ Duchenne only** | 4-dim Duchenne smiles features |
| 3 | **+ Incongruity only** | 3-dim incongruity-resolution features |
| 4 | **+ ToM only** | 3-dim Theory of Mind features |
| 5 | **Full model** | All 27 biosemotic dims |

### Metrics
- **F1** — token-level binary F1
- **IoU-F1** — sequence-level IoU-based F1
- **Precision / Recall**
- **Per-language breakdown** — English vs Mandarin vs Hindi

## 1. Setup & Installation

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers>=4.36', 'scikit-learn>=1.3', 'numpy>=1.24',
                'pandas>=2.0', 'matplotlib>=3.7', 'seaborn>=0.12', 'tqdm'], check=True)
print('✓ Dependencies installed')

In [ ]:
import os, json, time, gc
from pathlib import Path
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['HF_HUB_DISABLE_SYMLINKS'] = '1'

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, AutoConfig, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, precision_recall_fscore_support
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

MODEL_NAME = 'FacebookAI/xlm-roberta-base'
MAX_LENGTH = 256
BATCH_SIZE = 12
EPOCHS = 10
LR = 2e-5
POS_WEIGHT = 5.0
AUX_WEIGHT = 0.3
LABEL_SMOOTHING = 0.1
DROPOUT = 0.2
WEIGHT_DECAY = 0.02
PATIENCE = 3
UNFREEZE_LAYERS = 4

DATA_DIR = Path('data/final_merged_10k')
RESULTS_PATH = 'docs/ablation_results.json'
print(f'Data dir: {DATA_DIR}')

## 2. Data Download & Exploration

In [ ]:
import urllib.request

BASE_URL = 'https://github.com/Das-rebel/ChuckleNet/releases/download/v0.1-data'

for split in ['train', 'valid', 'test']:
    fpath = DATA_DIR / f'{split}.jsonl'
    if not fpath.exists():
        print(f'Downloading {split}.jsonl...')
        urllib.request.urlretrieve(f'{BASE_URL}/{split}.jsonl', fpath)
        print(f'  ✓ {split}.jsonl saved')
    else:
        print(f'  ✓ {split}.jsonl already exists')

for split in ['train', 'valid', 'test']:
    n = sum(1 for _ in open(DATA_DIR / f'{split}.jsonl'))
    print(f'{split}: {n:,} examples')

In [ ]:
def load_split(path):
    examples = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    examples.append(json.loads(line))
                except:
                    pass
    return examples

train_data = load_split(DATA_DIR / 'train.jsonl')
valid_data = load_split(DATA_DIR / 'valid.jsonl')
test_data  = load_split(DATA_DIR / 'test.jsonl')

print(f'Train: {len(train_data):,} | Valid: {len(valid_data):,} | Test: {len(test_data):,}')

sample = train_data[0]
BIOSEMOTIC_KEYS = [k for k in sample if k not in [
    'words', 'labels', 'language', 'metadata', 'example_id',
    'cue_bucket_ids', 'comedian_id', 'show_id', 'label',
    'is_sentence_level', 'tom_character_interaction_pattern'
]]
print(f'Biosemotic dims ({len(BIOSEMOTIC_KEYS)}): {BIOSEMOTIC_KEYS}')

In [ ]:
from collections import Counter

lang_train = Counter(d['language'] for d in train_data)
lang_valid = Counter(d['language'] for d in valid_data)
lang_test  = Counter(d['language'] for d in test_data)

print('=== Language Distribution ===')
print(f"{'Language':<8} {'Train':>7} {'Valid':>7} {'Test':>7}")
print('-' * 32)
for lang in sorted(set(lang_train) | set(lang_valid) | set(lang_test)):
    t = lang_train.get(lang, 0)
    v = lang_valid.get(lang, 0)
    te = lang_test.get(lang, 0)
    print(f'{lang:<8} {t:>7,} {v:>7,} {te:>7,}')

laugh_counts = Counter()
for d in train_data:
    for lbl in d.get('labels', []):
        laugh_counts[lbl] += 1
print(f'\nNon-laughter (0): {laugh_counts[0]:,}')
print(f'Laughter (1): {laugh_counts[1]:,}')
print(f'Imbalance ratio: {laugh_counts[0]/laugh_counts[1]:.1f}:1')

In [ ]:
import pandas as pd

feature_stats = {k: [] for k in BIOSEMOTIC_KEYS}
for d in train_data:
    for k in BIOSEMOTIC_KEYS:
        feature_stats[k].append(d.get(k, 0.5))

df_stats = pd.DataFrame({
    'Feature': BIOSEMOTIC_KEYS,
    'Mean': [np.mean(feature_stats[k]) for k in BIOSEMOTIC_KEYS],
    'Std':  [np.std(feature_stats[k]) for k in BIOSEMOTIC_KEYS],
    'Min':  [np.min(feature_stats[k]) for k in BIOSEMOTIC_KEYS],
    'Max':  [np.max(feature_stats[k]) for k in BIOSEMOTIC_KEYS],
})
df_stats = df_stats.sort_values('Mean', ascending=False).reset_index(drop=True)
print('=== Biosemotic Feature Statistics ===')
print(df_stats.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

langs = list(lang_train.keys())
counts = [lang_train[l] for l in langs]
colors_map = {'en': '#4C78A8', 'zh': '#F59E0B', 'hi': '#E45756'}
bar_colors = [colors_map.get(l, '#888888') for l in langs]

axes[0].bar(langs, counts, color=bar_colors, edgecolor='white', linewidth=0.7)
axes[0].set_xlabel('Language')
axes[0].set_ylabel('Number of Examples')
axes[0].set_title('Training Data: Language Distribution')
for i, (l, c) in enumerate(zip(langs, counts)):
    axes[0].text(i, c + 50, str(c), ha='center', va='bottom', fontweight='bold')

axes[1].pie(counts, labels=[f'{l}\n({c:,})' for l, c in zip(langs, counts)],
            colors=bar_colors, autopct='%1.1f%%', startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 1})
axes[1].set_title('Training Data: Language Proportions')

plt.tight_layout()
plt.savefig('docs/language_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Saved: docs/language_distribution.png')

In [ ]:
bio_df = pd.DataFrame(feature_stats)
corr = bio_df.corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax, linewidths=0.5, annot_kws={'size': 7})
ax.set_title('Biosemotic Feature Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.savefig('docs/biosemotic_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Saved: docs/biosemotic_correlation.png')

## 3. Dataset & Model Definitions

In [ ]:
class BiosemoticDataset(Dataset):
    def __init__(self, data_list, tokenizer, max_length=256):
        self.examples = []
        print(f'  Pre-tokenizing {len(data_list)} examples...')
        for idx, ex in enumerate(tqdm(data_list, desc='Tokenizing')):
            words = [str(w) for w in ex.get('words', [])]
            if not words:
                continue
            labels = ex.get('labels', [0] * len(words))

            enc = tokenizer(words, is_split_into_words=True,
                          truncation=True, max_length=max_length,
                          padding='max_length', return_tensors='pt')
            input_ids = enc['input_ids'].squeeze(0)
            attention_mask = enc['attention_mask'].squeeze(0)
            word_ids = enc.word_ids(batch_index=0)

            tok_labels = []
            prev = None
            for wid in word_ids:
                if wid is None:
                    tok_labels.append(-100)
                elif wid != prev and wid < len(labels):
                    tok_labels.append(labels[wid])
                    prev = wid
                else:
                    tok_labels.append(-100)

            def _s(field, default=0.5):
                v = ex.get(field, default)
                return float(v) if v is not None else default

            # Duchenne (4)
            duchenne = torch.tensor([
                _s('duchenne_joy_intensity'),
                _s('duchenne_genuine_humor_probability'),
                _s('duchenne_spontaneous_laughter_markers'),
                _s('duchenne_setup_punchline_structure'),
            ], dtype=torch.float32)

            # Incongruity (3)
            incongruity = torch.tensor([
                _s('incongruity_expectation_violation_score'),
                _s('incongruity_humor_complexity_score'),
                _s('incongruity_resolution_time'),
            ], dtype=torch.float32)

            # ToM (1 cls + 3 reg)
            intent_map = {'humor_expression': 0, 'playful_banter': 1, 'storytelling': 2}
            tom_intent = torch.tensor(
                intent_map.get(ex.get('tom_speaker_intent_label', ''), 0),
                dtype=torch.long)
            tom_reg = torch.tensor([
                _s('tom_speaker_intent_confidence'),
                _s('tom_audience_persience_score'),
                _s('tom_social_context_humor_score'),
            ], dtype=torch.float32)

            # Cue buckets (7)
            cue_ids = ex.get('cue_bucket_ids', [0] * 7)
            cue_buckets = torch.tensor((cue_ids + [0] * 7)[:7], dtype=torch.long)

            # Structural (3)
            structural = torch.tensor([
                _s('clause_boundary_ratio', 0.33),
                _s('punchline_zone_ratio', 0.33),
                _s('setup_zone_ratio', 0.33),
            ], dtype=torch.float32)

            # Linguistic (4)
            linguistic = torch.tensor([
                _s('filler_token_ratio', 0.1),
                _s('negation_ratio', 0.1),
                _s('exclamation_ratio', 0.1),
                _s('quoted_speech_ratio', 0.1),
            ], dtype=torch.float32)

            # Metadata hashes (6)
            def _hash_cat(field, n_bins=6):
                return hash(str(ex.get(field, 'unknown'))) % n_bins
            metadata = torch.tensor([
                _hash_cat('language'),
                _hash_cat('comedian_id'),
                _hash_cat('show_id'),
                _hash_cat('tom_character_interaction_pattern'),
                0, 0,
            ], dtype=torch.long)

            self.examples.append({
                'input_ids': input_ids,
                'attention_mask': attention_mask,
                'token_labels': torch.tensor(tok_labels, dtype=torch.long),
                'duchenne': duchenne,
                'incongruity': incongruity,
                'tom_intent': tom_intent,
                'tom_reg': tom_reg,
                'cue_buckets': cue_buckets,
                'structural': structural,
                'linguistic': linguistic,
                'metadata': metadata,
                'language': ex.get('language', 'en'),
            })
        print(f'  Done: {len(self.examples)} examples')

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return {k: v.clone() if isinstance(self.examples[idx][k], torch.Tensor) else self.examples[idx][k]
                for k in self.examples[idx]}

In [ ]:
def collate_fn(batch):
    result = {}
    for k in batch[0]:
        if isinstance(batch[0][k], torch.Tensor):
            result[k] = torch.stack([b[k] for b in batch])
        else:
            result[k] = [b[k] for b in batch]
    return result

In [ ]:
class BiosemoticModel(nn.Module):
    def __init__(self, model_name=MODEL_NAME,
                 disable_duchenne=False, disable_incongruity=False,
                 disable_tom=False, disable_cue=False,
                 disable_structural=False, disable_linguistic=False,
                 disable_metadata=False,
                 dropout=DROPOUT):
        super().__init__()
        self.flags = {
            'duchenne': disable_duchenne,
            'incongruity': disable_incongruity,
            'tom': disable_tom,
            'cue': disable_cue,
            'structural': disable_structural,
            'linguistic': disable_linguistic,
            'metadata': disable_metadata,
        }

        config = AutoConfig.from_pretrained(model_name)
        self.backbone = AutoModel.from_pretrained(model_name, config=config)
        hidden = config.hidden_size

        self.laughter_head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, 2)
        )

        self.duchenne_head    = nn.Linear(hidden, 4) if not disable_duchenne else None
        self.incongruity_head = nn.Linear(hidden, 3) if not disable_incongruity else None
        self.tom_intent_head  = nn.Linear(hidden, 3) if not disable_tom else None
        self.tom_reg_head    = nn.Linear(hidden, 3) if not disable_tom else None
        self.cue_head        = nn.Linear(hidden, 7) if not disable_cue else None
        self.structural_head = nn.Linear(hidden, 3) if not disable_structural else None
        self.linguistic_head = nn.Linear(hidden, 4) if not disable_linguistic else None
        self.metadata_head   = nn.Linear(hidden, 6) if not disable_metadata else None

    def forward(self, input_ids, attention_mask):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        seq = out.last_hidden_state
        result = {'laughter_logits': self.laughter_head(seq)}
        if self.duchenne_head:    result['duchenne']    = self.duchenne_head(seq[:, 0])
        if self.incongruity_head: result['incongruity'] = self.incongruity_head(seq[:, 0])
        if self.tom_intent_head:  result['tom_intent']  = self.tom_intent_head(seq[:, 0])
        if self.tom_reg_head:     result['tom_reg']     = self.tom_reg_head(seq[:, 0])
        if self.cue_head:        result['cue']         = self.cue_head(seq[:, 0])
        if self.structural_head: result['structural']  = self.structural_head(seq[:, 0])
        if self.linguistic_head: result['linguistic']   = self.linguistic_head(seq[:, 0])
        if self.metadata_head:   result['metadata']    = self.metadata_head(seq[:, 0])
        return result

In [ ]:
def compute_loss(outputs, batch, pos_weight=POS_WEIGHT, aux_weight=AUX_WEIGHT,
                label_smoothing=LABEL_SMOOTHING):
    device = outputs['laughter_logits'].device
    losses = {}

    logits = outputs['laughter_logits']
    labels = batch['token_labels'].to(device)
    mask = (labels != -100)
    if mask.sum() > 0:
        cw = torch.tensor([1.0, pos_weight], device=device)
        losses['laughter'] = F.cross_entropy(
            logits.view(-1, 2)[mask.view(-1)],
            labels.view(-1)[mask.view(-1)],
            weight=cw,
            label_smoothing=label_smoothing)

    if 'duchenne' in outputs:
        losses['duchenne'] = F.mse_loss(outputs['duchenne'], batch['duchenne'].to(device))
    if 'incongruity' in outputs:
        losses['incongruity'] = F.mse_loss(outputs['incongruity'], batch['incongruity'].to(device))
    if 'tom_intent' in outputs:
        losses['tom_intent'] = F.cross_entropy(outputs['tom_intent'], batch['tom_intent'].to(device))
    if 'tom_reg' in outputs:
        losses['tom_reg'] = F.mse_loss(outputs['tom_reg'], batch['tom_reg'].to(device))
    if 'cue' in outputs:
        losses['cue'] = F.cross_entropy(outputs['cue'], batch['cue_buckets'].to(device))
    if 'structural' in outputs:
        losses['structural'] = F.mse_loss(outputs['structural'], batch['structural'].to(device))
    if 'linguistic' in outputs:
        losses['linguistic'] = F.mse_loss(outputs['linguistic'], batch['linguistic'].to(device))
    if 'metadata' in outputs:
        losses['metadata'] = F.cross_entropy(outputs['metadata'], batch['metadata'].to(device))

    total = losses['laughter']
    aux_keys = [k for k in losses if k != 'laughter']
    if aux_keys:
        total = total + aux_weight * sum(losses[k] for k in aux_keys) / len(aux_keys)
    return total, losses


def _get_span(label_array):
    ones = np.where(label_array == 1)[0]
    if len(ones) == 0:
        return (0, 0)
    return (ones[0], ones[-1] + 1)


def _span_iou(pred_span, true_span):
    p_start, p_end = pred_span
    t_start, t_end = true_span
    if p_start == p_end == 0 and t_start == t_end == 0:
        return 1.0
    if p_start == p_end or t_start == t_end:
        return 0.0
    inter = max(0, min(p_end, t_end) - max(p_start, t_start))
    union = max(p_end, t_end) - min(p_start, t_start)
    return inter / union if union > 0 else 0.0


@torch.no_grad()
def evaluate_model(model, loader, device, pos_weight=POS_WEIGHT,
                   aux_weight=AUX_WEIGHT, label_smoothing=LABEL_SMOOTHING,
                   compute_iou=True):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0.0
    lang_preds, lang_labels = {}, {}

    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        outputs = model(input_ids, attention_mask)
        loss, _ = compute_loss(outputs, batch, pos_weight, aux_weight, label_smoothing)
        total_loss += loss.item()

        labels = batch['token_labels'].to(device)
        mask = (labels != -100)
        preds = outputs['laughter_logits'].argmax(dim=-1)
        all_preds.extend(preds[mask].cpu().tolist())
        all_labels.extend(labels[mask].cpu().tolist())

        for i, lang in enumerate(batch.get('language', [])):
            lang_mask_i = mask[i]
            lang_preds.setdefault(lang, []).extend(preds[i][lang_mask_i].cpu().tolist())
            lang_labels.setdefault(lang, []).extend(labels[i][lang_mask_i].cpu().tolist())

    avg_loss = total_loss / len(loader)
    f1 = f1_score(all_labels, all_preds, pos_label=1, zero_division=0)
    p, r, _, _ = precision_recall_fscore_support(all_labels, all_preds, pos_label=1, zero_division=0)
    precision, recall = float(p[1]), float(r[1])

    iou_f1 = None
    if compute_iou:
        total_iou, n_seqs = 0.0, 0
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            outputs = model(input_ids, attention_mask)
            preds = outputs['laughter_logits'].argmax(dim=-1)
            for i in range(preds.shape[0]):
                pred_span = _get_span(preds[i].cpu().numpy())
                true_span = _get_span(batch['token_labels'][i].numpy())
                total_iou += _span_iou(pred_span, true_span)
                n_seqs += 1
        avg_iou = total_iou / max(n_seqs, 1)
        iou_f1 = 2 * avg_iou / (avg_iou + 1.0) if avg_iou > 0 else 0.0

    lang_metrics = {}
    for lang in lang_preds:
        lp, lr_l = lang_preds[lang], lang_labels[lang]
        if sum(lr_l) > 0:
            lf1 = f1_score(lr_l, lp, pos_label=1, zero_division=0)
            lp_r, lr_r, _, _ = precision_recall_fscore_support(lr_l, lp, pos_label=1, zero_division=0)
            lang_metrics[lang] = {'f1': lf1, 'precision': float(lp_r[1]), 'recall': float(lr_r[1]), 'n_examples': len(lp)}

    return {'loss': avg_loss, 'f1': f1, 'precision': precision, 'recall': recall,
            'iou_f1': iou_f1, 'lang_metrics': lang_metrics}

## 4. Training Loop

In [ ]:
def run_experiment(name, train_data, valid_data, test_data,
                  pos_weight=POS_WEIGHT, epochs=EPOCHS, lr=LR,
                  batch_size=BATCH_SIZE, aux_weight=AUX_WEIGHT,
                  patience=PATIENCE, unfreeze_layers=UNFREEZE_LAYERS,
                  dropout=DROPOUT, label_smoothing=LABEL_SMOOTHING,
                  weight_decay=WEIGHT_DECAY,
                  disable_duchenne=False, disable_incongruity=False,
                  disable_tom=False, disable_cue=False,
                  disable_structural=False, disable_linguistic=False,
                  disable_metadata=False,
                  compute_iou=True,
                  save_path=RESULTS_PATH):
    print(f'\n{'='*65}')
    print(f' {name}')
    print(f' pw={pos_weight} bs={batch_size} lr={lr} drop={dropout}')
    print(f' aux={aux_weight} smooth={label_smoothing} wd={weight_decay}')
    disabled = [k for k, v in {
        'duchenne': disable_duchenne, 'incongruity': disable_incongruity,
        'tom': disable_tom, 'cue': disable_cue,
        'structural': disable_structural, 'linguistic': disable_linguistic,
        'metadata': disable_metadata
    }.items() if v]
    if disabled:
        print(f' Disabled: {disabled}')
    print(f'{'='*65}')

    t0 = time.time()
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    train_ds = BiosemoticDataset(train_data, tokenizer, MAX_LENGTH)
    valid_ds = BiosemoticDataset(valid_data, tokenizer, MAX_LENGTH)
    test_ds  = BiosemoticDataset(test_data,  tokenizer, MAX_LENGTH)

    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                          collate_fn=collate_fn, num_workers=2, pin_memory=True)
    valid_dl = DataLoader(valid_ds, batch_size=batch_size*2, shuffle=False,
                          collate_fn=collate_fn, num_workers=2, pin_memory=True)
    test_dl  = DataLoader(test_ds,  batch_size=batch_size*2, shuffle=False,
                          collate_fn=collate_fn, num_workers=2, pin_memory=True)

    model = BiosemoticModel(
        dropout=dropout,
        disable_duchenne=disable_duchenne,
        disable_incongruity=disable_incongruity,
        disable_tom=disable_tom,
        disable_cue=disable_cue,
        disable_structural=disable_structural,
        disable_linguistic=disable_linguistic,
        disable_metadata=disable_metadata,
    ).to(device)

    optimizer = torch.optim.AdamW([
        {'params': model.backbone.parameters(),       'lr': lr,        'weight_decay': weight_decay},
        {'params': model.laughter_head.parameters(), 'lr': lr * 5,   'weight_decay': weight_decay},
    ])

    total_steps = len(train_dl) * epochs
    warmup_steps = int(0.1 * total_steps)
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

    best_f1 = 0.0
    best_state = None
    wait = 0

    for epoch in range(epochs):
        for p in model.backbone.parameters():
            p.requires_grad = False
        if 0 < unfreeze_layers < 12:
            for layer in model.backbone.encoder.layer[-unfreeze_layers:]:
                for p in layer.parameters():
                    p.requires_grad = True
        elif unfreeze_layers >= 12:
            for p in model.backbone.parameters():
                p.requires_grad = True
        for p in model.laughter_head.parameters():
            p.requires_grad = True

        model.train()
        epoch_loss = 0.0
        for batch in tqdm(train_dl, desc=f'E{epoch+1} train', leave=False):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            outputs = model(input_ids, attention_mask)
            loss, _ = compute_loss(outputs, batch, pos_weight, aux_weight, label_smoothing)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            epoch_loss += loss.item()

        val = evaluate_model(model, valid_dl, device, pos_weight, aux_weight,
                            label_smoothing, compute_iou=False)
        improved = val['f1'] > best_f1
        if improved:
            best_f1 = val['f1']
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1

        elapsed = (time.time() - t0) / 60
        print(f'  E{epoch+1:2d}/{epochs} loss={epoch_loss/len(train_dl):.4f} '
              f'VaF1={val["f1"]:.4f} best={best_f1:.4f} '
              f'{"★" if improved else ""} [{elapsed:.1f}m]')

        if wait >= patience:
            print(f'  Early stop at epoch {epoch+1}')
            break

    if best_state:
        model.load_state_dict(best_state)
    test_metrics = evaluate_model(model, test_dl, device, pos_weight, aux_weight,
                                 label_smoothing, compute_iou=compute_iou)

    print(f'\n  TEST: F1={test_metrics["f1"]:.4f} P={test_metrics["precision"]:.4f} '
          f'R={test_metrics["recall"]:.4f} IoU-F1={test_metrics.get("iou_f1", 0):.4f}')

    print(f'\n  Per-language test results:')
    for lang, m in sorted(test_metrics['lang_metrics'].items()):
        print(f'    {lang}: F1={m["f1"]:.4f} P={m["precision"]:.4f} R={m["recall"]:.4f} (n={m["n_examples"]})')

    del model, best_state, train_ds, valid_ds, test_ds
    gc.collect()
    torch.cuda.empty_cache()

    result = {
        'name': name,
        'disabled': disabled,
        'val_f1': best_f1,
        'test_f1': test_metrics['f1'],
        'test_precision': test_metrics['precision'],
        'test_recall': test_metrics['recall'],
        'test_iou_f1': test_metrics.get('iou_f1', 0.0),
        'test_lang_metrics': {k: {kk: float(vv) for kk, vv in v.items()}
                              for k, v in test_metrics['lang_metrics'].items()},
        'time_min': round((time.time() - t0) / 60),
        'config': {
            'pos_weight': pos_weight, 'batch_size': batch_size, 'lr': lr,
            'dropout': dropout, 'label_smoothing': label_smoothing,
            'weight_decay': weight_decay, 'aux_weight': aux_weight
        }
    }

    _save_results(result, save_path)
    return result


def _save_results(result, path):
    results = []
    if Path(path).exists():
        with open(path) as f:
            data = json.load(f)
            results = data.get('results', [])

    results.append(result)
    output = {
        'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
        'total_experiments': 5,
        'completed': len(results),
        'results': results
    }
    with open(path, 'w') as f:
        json.dump(output, f, indent=2)
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=True)
        with open(f'/content/drive/MyDrive/{Path(path).name}', 'w') as f:
            json.dump(output, f, indent=2)
        print(f'  ✓ Saved to Drive + local ({len(results)}/5)')
    except:
        print(f'  ✓ Saved locally ({len(results)}/5)')

## 5. Run Ablation Study

In [ ]:
print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f'✓ Tokenizer loaded: {MODEL_NAME}')

In [ ]:
# EXP 1: Baseline XLM-R (no biosemotic features)
result_baseline = run_experiment(
    name='Baseline_XLMR_NoBiosemotic',
    train_data=train_data, valid_data=valid_data, test_data=test_data,
    pos_weight=5.0, epochs=EPOCHS, lr=LR,
    batch_size=BATCH_SIZE, aux_weight=0.0,
    patience=PATIENCE, unfreeze_layers=UNFREEZE_LAYERS,
    dropout=DROPOUT, label_smoothing=LABEL_SMOOTHING,
    weight_decay=WEIGHT_DECAY,
    disable_duchenne=True, disable_incongruity=True,
    disable_tom=True, disable_cue=True,
    disable_structural=True, disable_linguistic=True,
    disable_metadata=True,
    compute_iou=True,
    save_path=RESULTS_PATH
)

In [ ]:
# EXP 2: XLM-R + Duchenne features only
result_duchenne = run_experiment(
    name='XLMR_Duchenne_Only',
    train_data=train_data, valid_data=valid_data, test_data=test_data,
    pos_weight=5.0, epochs=EPOCHS, lr=LR,
    batch_size=BATCH_SIZE, aux_weight=0.3,
    patience=PATIENCE, unfreeze_layers=UNFREEZE_LAYERS,
    dropout=DROPOUT, label_smoothing=LABEL_SMOOTHING,
    weight_decay=WEIGHT_DECAY,
    disable_duchenne=False, disable_incongruity=True,
    disable_tom=True, disable_cue=True,
    disable_structural=True, disable_linguistic=True,
    disable_metadata=True,
    compute_iou=True,
    save_path=RESULTS_PATH
)

In [ ]:
# EXP 3: XLM-R + Incongruity features only
result_incongruity = run_experiment(
    name='XLMR_Incongruity_Only',
    train_data=train_data, valid_data=valid_data, test_data=test_data,
    pos_weight=5.0, epochs=EPOCHS, lr=LR,
    batch_size=BATCH_SIZE, aux_weight=0.3,
    patience=PATIENCE, unfreeze_layers=UNFREEZE_LAYERS,
    dropout=DROPOUT, label_smoothing=LABEL_SMOOTHING,
    weight_decay=WEIGHT_DECAY,
    disable_duchenne=True, disable_incongruity=False,
    disable_tom=True, disable_cue=True,
    disable_structural=True, disable_linguistic=True,
    disable_metadata=True,
    compute_iou=True,
    save_path=RESULTS_PATH
)

In [ ]:
# EXP 4: XLM-R + ToM features only
result_tom = run_experiment(
    name='XLMR_ToM_Only',
    train_data=train_data, valid_data=valid_data, test_data=test_data,
    pos_weight=5.0, epochs=EPOCHS, lr=LR,
    batch_size=BATCH_SIZE, aux_weight=0.3,
    patience=PATIENCE, unfreeze_layers=UNFREEZE_LAYERS,
    dropout=DROPOUT, label_smoothing=LABEL_SMOOTHING,
    weight_decay=WEIGHT_DECAY,
    disable_duchenne=True, disable_incongruity=True,
    disable_tom=False, disable_cue=True,
    disable_structural=True, disable_linguistic=True,
    disable_metadata=True,
    compute_iou=True,
    save_path=RESULTS_PATH
)

In [ ]:
# EXP 5: XLM-R + ALL biosemotic features (full model)
result_full = run_experiment(
    name='XLMR_FullBiosemotic_AllDims',
    train_data=train_data, valid_data=valid_data, test_data=test_data,
    pos_weight=5.0, epochs=EPOCHS, lr=LR,
    batch_size=BATCH_SIZE, aux_weight=0.3,
    patience=PATIENCE, unfreeze_layers=UNFREEZE_LAYERS,
    dropout=DROPOUT, label_smoothing=LABEL_SMOOTHING,
    weight_decay=WEIGHT_DECAY,
    disable_duchenne=False, disable_incongruity=False,
    disable_tom=False, disable_cue=False,
    disable_structural=False, disable_linguistic=False,
    disable_metadata=False,
    compute_iou=True,
    save_path=RESULTS_PATH
)

## 6. Results Summary

In [ ]:
with open(RESULTS_PATH) as f:
    full_output = json.load(f)
results = full_output['results']
print(f'Total experiments: {len(results)}')

rows = []
for r in results:
    rows.append({
        'Experiment': r['name'],
        'Disabled': ','.join(r.get('disabled', [])) or 'None',
        'Val F1': r['val_f1'],
        'Test F1': r['test_f1'],
        'Test Precision': r['test_precision'],
        'Test Recall': r['test_recall'],
        'Test IoU-F1': r.get('test_iou_f1', 0.0),
        'Time (min)': r['time_min'],
    })

df_results = pd.DataFrame(rows)
df_results = df_results.sort_values('Test F1', ascending=False).reset_index(drop=True)
df_results.index = df_results.index + 1
print('\n=== ABLATION STUDY RESULTS ===')
print(df_results.to_string())

In [ ]:
baseline = next((r for r in results if 'Baseline' in r['name']), None)
full     = next((r for r in results if 'FullBiosemotic' in r['name']), None)

print('=== KEY INSIGHTS ===\n')

if baseline and full:
    delta = full['test_f1'] - baseline['test_f1']
    delta_iou = full.get('test_iou_f1', 0) - baseline.get('test_iou_f1', 0)
    print(f'Baseline → Full Biosemotic:')
    print(f'  F1:     {baseline["test_f1"]:.4f} → {full["test_f1"]:.4f} (Δ={delta:+.4f})')
    print(f'  IoU-F1: {baseline.get("test_iou_f1",0):.4f} → {full.get("test_iou_f1",0):.4f} (Δ={delta_iou:+.4f})')
    print(f'  Biosemotic {"HELPS" if delta > 0 else "does NOT help"}\n')

baseline_f1 = baseline['val_f1'] if baseline else 0
feature_impact = []
for r in results:
    if 'Only' in r['name']:
        dim_name = r['name'].split('_')[1]
        impact = baseline_f1 - r['val_f1']
        feature_impact.append((dim_name, r['val_f1'], impact))

print('Feature Importance (Δ F1 when disabled individually):')
print(f"{'Feature':<15} {'Val F1':>8} {'Δ Impact':>10}")
print('-' * 35)
for name, f1, imp in sorted(feature_impact, key=lambda x: x[2], reverse=True):
    arrow = '↑' if imp > 0 else '↓'
    print(f'{name:<15} {f1:>8.4f} {arrow}{abs(imp):>9.4f}')

## 7. Visualizations

In [ ]:
# Visualization 1: F1 comparison bar chart
fig, ax = plt.subplots(figsize=(12, 6))

names = [r['name'].replace('XLMR_', '').replace('_Only', '').replace('_AllDims', '_Full') for r in results]
test_f1s = [r['test_f1'] for r in results]
val_f1s  = [r['val_f1'] for r in results]
iou_f1s  = [r.get('test_iou_f1', 0) for r in results]

x = np.arange(len(results))
width = 0.25

bars1 = ax.bar(x - width, val_f1s,  width, label='Val F1',       color='#4C78A8', edgecolor='white')
bars2 = ax.bar(x,         test_f1s, width, label='Test F1',      color='#F59E0B', edgecolor='white')
bars3 = ax.bar(x + width, iou_f1s,  width, label='Test IoU-F1',  color='#E45756', edgecolor='white')

ax.set_xlabel('Experiment')
ax.set_ylabel('Score')
ax.set_title('V8.1 Ablation Study: F1 & IoU-F1 Comparison')
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=30, ha='right', fontsize=9)
ax.legend(fontsize=10)
ax.set_ylim(0, 1.0)

for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=7)

plt.tight_layout()
plt.savefig('docs/ablation_f1_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Saved: docs/ablation_f1_comparison.png')

In [ ]:
# Visualization 2: Precision-Recall scatter
fig, ax = plt.subplots(figsize=(8, 8))

colors_map = {
    'Baseline_XLMR': '#888888',
    'Duchenne': '#4C78A8',
    'Incongruity': '#F59E0B',
    'ToM': '#E45756',
    'FullBiosemotic': '#54A24B',
}

for r in results:
    short = r['name'].replace('XLMR_', '').replace('_Only', '').replace('_AllDims', '_Full')
    color = next((c for k, c in colors_map.items() if k in short), '#888888')
    ax.scatter(r['test_recall'], r['test_precision'],
               s=200, c=color, edgecolors='white', linewidth=1.5, zorder=3)
    ax.annotate(short, (r['test_recall'], r['test_precision']),
                textcoords='offset points', xytext=(8, 8), fontsize=8)

ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision vs Recall — Ablation Study')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3)

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, edgecolor='white', label=k)
                   for k, c in colors_map.items() if any(k in r['name'] for r in results)]
ax.legend(handles=legend_elements, fontsize=9, loc='lower left')

plt.tight_layout()
plt.savefig('docs/ablation_precision_recall.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Saved: docs/ablation_precision_recall.png')

In [ ]:
# Visualization 3: Language-specific performance heatmap
lang_metrics = {}
for r in results:
    for lang, m in r.get('test_lang_metrics', {}).items():
        lang_metrics.setdefault(lang, {})
        short = r['name'].replace('XLMR_', '').replace('_Only', '').replace('_AllDims', '_Full')
        lang_metrics[lang][short] = m

if lang_metrics:
    n_langs = len(lang_metrics)
    fig, axes = plt.subplots(1, n_langs, figsize=(5 * n_langs, 5))
    if n_langs == 1:
        axes = [axes]
    else:
        axes = axes.flatten()

    for ax, (lang, exp_metrics) in zip(axes, lang_metrics.items()):
        rows = []
        for exp, m in exp_metrics.items():
            rows.append([exp, m['f1'], m['precision'], m['recall']])
        ld = pd.DataFrame(rows, columns=['Experiment', 'F1', 'Precision', 'Recall'])
        ld = ld.sort_values('F1', ascending=False)
        sns.heatmap(ld[['F1', 'Precision', 'Recall']].values, ax=ax, annot=True,
                    fmt='.3f', cmap='YlOrRd', vmin=0, vmax=1,
                    xticklabels=['F1', 'Precision', 'Recall'],
                    yticklabels=ld['Experiment'].tolist(),
                    linewidths=0.5)
        ax.set_title(f'{lang.upper()} Test Performance', fontsize=12)
        ax.set_yticklabels(ld['Experiment'].tolist(), rotation=0, fontsize=8)

    plt.tight_layout()
    plt.savefig('docs/ablation_language_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✓ Saved: docs/ablation_language_heatmap.png')

In [ ]:
# Visualization 4: Feature contribution impact bar chart
fig, ax = plt.subplots(figsize=(10, 5))

baseline_f1 = baseline['test_f1'] if baseline else 0
feature_impact = []
for r in results:
    if 'Only' in r['name']:
        dim = r['name'].split('_')[1]
        delta = r['test_f1'] - baseline_f1
        feature_impact.append({'feature': dim, 'f1': r['test_f1'], 'delta': delta})

if feature_impact:
    feature_impact.sort(key=lambda x: x['delta'], reverse=True)
    features = [x['feature'] for x in feature_impact]
    deltas = [x['delta'] for x in feature_impact]
    colors = ['#54A24B' if d >= 0 else '#E45756' for d in deltas]
    bars = ax.barh(features, deltas, color=colors, edgecolor='white', linewidth=0.7)
    ax.axvline(x=0, color='black', linewidth=0.8)
    ax.set_xlabel('Delta F1 vs Baseline')
    ax.set_title('Biosemotic Feature Impact: Δ Test F1 when added individually')
    for bar, delta in zip(bars, deltas):
        x_pos = delta + 0.002 if delta >= 0 else delta - 0.002
        ha = 'left' if delta >= 0 else 'right'
        ax.text(x_pos, bar.get_y() + bar.get_height()/2,
                f'{delta:+.4f}', va='center', ha=ha, fontsize=9)
    plt.tight_layout()
    plt.savefig('docs/ablation_feature_impact.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✓ Saved: docs/ablation_feature_impact.png')

In [ ]:
# Visualization 5: All metrics grouped bar chart
fig, ax = plt.subplots(figsize=(14, 6))

names = [r['name'].replace('XLMR_', '').replace('_Only', '').replace('_AllDims', '_Full') for r in results]
x = np.arange(len(results))
width = 0.2

ax.bar(x - 1.5*width, [r['val_f1'] for r in results], width, label='Val F1',    color='#4C78A8')
ax.bar(x - 0.5*width, [r['test_f1'] for r in results], width, label='Test F1', color='#F59E0B')
ax.bar(x + 0.5*width, [r['test_precision'] for r in results], width, label='Precision', color='#54A24B')
ax.bar(x + 1.5*width, [r['test_recall'] for r in results], width, label='Recall', color='#E45756')

ax.set_xlabel('Experiment')
ax.set_ylabel('Score')
ax.set_title('V8.1 Ablation: All Metrics Comparison')
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=30, ha='right', fontsize=9)
ax.legend(fontsize=9)
ax.set_ylim(0, 1.0)

plt.tight_layout()
plt.savefig('docs/ablation_all_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Saved: docs/ablation_all_metrics.png')

## 8. Final Summary

In [ ]:
print('\n' + '='*70)
print(' V8.1 ABLATION STUDY — FINAL RESULTS')
print('='*70)
print(f"\n{'Rank':<5} {'Experiment':<35} {'ValF1':>7} {'TeF1':>7} {'Prec':>7} {'Rec':>7} {'IoU-F1':>7} {'Time':>6}")
print('-'*90)
for i, r in enumerate(sorted(results, key=lambda x: x['test_f1'], reverse=True), 1):
    print(f"{i:<5} {r['name']:<35} {r['val_f1']:>7.4f} {r['test_f1']:>7.4f} "
          f"{r['test_precision']:>7.4f} {r['test_recall']:>7.4f} "
          f"{r.get('test_iou_f1',0):>7.4f} {r['time_min']:>5}m")

print('\n' + '='*70)
best  = sorted(results, key=lambda x: x['test_f1'], reverse=True)[0]
worst = sorted(results, key=lambda x: x['test_f1'])[0]
print(f'BEST:  {best["name"]} — Test F1={best["test_f1"]:.4f}')
print(f'WORST: {worst["name"]} — Test F1={worst["test_f1"]:.4f}')
print(f'Improvement: {(best["test_f1"] - worst["test_f1"]):.4f}')
print('='*70)
print(f'\nResults saved to: {RESULTS_PATH}')